In [1]:
# Базовые библиотеки
import os, sys, random, json, re
from typing import List, Dict, Optional, Tuple  # ← Tuple добавлен!
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer

# Отключаем параллелизм токенизаторов для воспроизводимости
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Фиксируем seed
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
set_seed(42)

# Определяем устройство
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except:
    DEVICE = "cpu"
print(f"Устройство: {DEVICE}")

Устройство: cpu


In [2]:
# Авто-установка пакетов, если не установлены
def ensure(pkg, imp=None):
    target = imp or pkg
    try:
        __import__(target)
        return True
    except:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        __import__(target)
        return True

FAISS_OK = ensure("faiss-cpu", "faiss")
ST_OK = ensure("sentence-transformers", "sentence_transformers")
print(f"FAISS: {FAISS_OK}, sentence-transformers: {ST_OK}")

C:\Users\MSI\Desktop\Artificial Intelligence Engineering\aie-group-3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FAISS: True, sentence-transformers: True


In [3]:
# Учебная база знаний по телекоммуникациям и сетям
documents: List[Dict[str, str]] = [
    {"doc_id": "net_01", "title": "Модель OSI", "text": "Модель OSI состоит из семи уровней: физический, канальный, сетевой, транспортный, сеансовый, представления и прикладной. Каждый уровень выполняет строго определённую функцию в передаче данных."},
    {"doc_id": "net_02", "title": "Стек протоколов TCP/IP", "text": "TCP/IP включает четыре уровня: прикладной, транспортный, интернет и доступа к сети. Протоколы TCP и IP обеспечивают надёжную доставку пакетов и маршрутизацию в глобальных сетях."},
    {"doc_id": "net_03", "title": "Ошибки передачи данных", "text": "При передаче данных возможны потери пакетов, искажение битов и задержки. Для обнаружения ошибок используются контрольные суммы, CRC и механизмы повторной передачи (ARQ)."},
    {"doc_id": "net_04", "title": "Маршрутизация в сетях", "text": "Маршрутизаторы анализируют заголовки пакетов и выбирают оптимальный путь на основе таблиц маршрутизации. Протоколы OSPF и BGP используются для динамического обновления маршрутов."},
    {"doc_id": "net_05", "title": "Качество обслуживания (QoS)", "text": "QoS позволяет приоритезировать трафик: голос и видео получают более высокий приоритет, чем файловые загрузки. Это снижает задержки и джиттер для чувствительных приложений."},
    {"doc_id": "net_06", "title": "Безопасность сетей", "text": "Межсетевые экраны, VPN и шифрование TLS защищают данные от перехвата и несанкционированного доступа. Аутентификация и авторизация контролируют доступ к ресурсам."},
    {"doc_id": "net_07", "title": "Беспроводные технологии", "text": "Wi-Fi, Bluetooth и сотовые сети используют радиоканалы. Помехи, затухание сигнала и коллизии снижают надёжность, поэтому применяются методы модуляции и множественного доступа."},
    {"doc_id": "net_08", "title": "Протоколы прикладного уровня", "text": "HTTP, DNS, SMTP и FTP работают на прикладном уровне модели OSI. Они определяют формат запросов и ответов для веб-серфинга, почты и передачи файлов."},
    {"doc_id": "net_09", "title": "Диагностика сетей", "text": "Утилиты ping, traceroute и tcpdump помогают выявлять проблемы: потерю пакетов, неправильную маршрутизацию или блокировку портов. Логи и метрики упрощают анализ инцидентов."},
    {"doc_id": "net_10", "title": "Масштабирование сетей", "text": "При росте нагрузки сети масштабируют горизонтально (добавление узлов) и вертикально (усиление оборудования). Балансировка нагрузки распределяет трафик между серверами."},
]

docs_df = pd.DataFrame(documents)
print(f"Документов в базе: {len(docs_df)}")
display(docs_df[["doc_id", "title"]])

Документов в базе: 10


,doc_id,title
0,net_01,Модель OSI
1,net_02,Стек протоколов TCP/IP
2,net_03,Ошибки передачи данных
3,net_04,Маршрутизация в сетях
4,net_05,Качество обслуживания (QoS)
5,net_06,Безопасность сетей
6,net_07,Беспроводные технологии
7,net_08,Протоколы прикладного уровня
8,net_09,Диагностика сетей
9,net_10,Масштабирование сетей


In [4]:
def chunk_text(text: str, chunk_size: int = 20, overlap: int = 5) -> List[str]:
    words = text.split()
    if chunk_size <= 0 or overlap >= chunk_size:
        raise ValueError("Некорректные параметры чанкинга")
    chunks, step = [], chunk_size - overlap
    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start+chunk_size])
        if chunk: chunks.append(chunk)
        if start + chunk_size >= len(words): break
    return chunks

def build_chunks_df(docs: List[Dict], chunk_size: int = 20, overlap: int = 5) -> pd.DataFrame:
    rows = []
    for doc in docs:
        for i, chunk in enumerate(chunk_text(doc["text"], chunk_size, overlap)):
            rows.append({"doc_id": doc["doc_id"], "title": doc["title"], "chunk_id": f"{doc['doc_id']}_c{i}", "chunk_text": chunk})
    return pd.DataFrame(rows)

chunks_df = build_chunks_df(documents, chunk_size=20, overlap=5)
print(f"Чанков после разбиения: {len(chunks_df)}")
display(chunks_df.head(3))

Чанков после разбиения: 18


,doc_id,title,chunk_id,chunk_text
0,net_01,Модель OSI,net_01_c0,Модель OSI состоит из семи уровней: физический...
1,net_01,Модель OSI,net_01_c1,уровень выполняет строго определённую функцию ...
2,net_02,Стек протоколов TCP/IP,net_02_c0,"TCP/IP включает четыре уровня: прикладной, тра..."


In [5]:
# Бэкенд для эмбеддингов
class EmbedBackend:
    def fit(self, texts: List[str]) -> np.ndarray: raise NotImplementedError
    def encode(self, texts: List[str]) -> np.ndarray: raise NotImplementedError

class STBackend(EmbedBackend):
    def __init__(self, device="cpu"):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=device)
    def fit(self, texts): return self.model.encode(texts, normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    def encode(self, texts): return self.model.encode(texts, normalize_embeddings=True, convert_to_numpy=True).astype("float32")

class TfidfBackend(EmbedBackend):
    def __init__(self): self.vec = TfidfVectorizer(ngram_range=(1,2))
    def fit(self, texts):
        X = self.vec.fit_transform(texts).toarray().astype("float32")
        return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    def encode(self, texts):
        X = self.vec.transform(texts).toarray().astype("float32")
        return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)

# Выбираем бэкенд
embedder = STBackend(DEVICE) if ST_OK else TfidfBackend()
print(f"Бэкенд эмбеддингов: {type(embedder).__name__}")

# Векторизуем чанки
chunk_vecs = embedder.fit(chunks_df["chunk_text"].tolist())
print(f"Размерность эмбеддингов: {chunk_vecs.shape}")

# Строим индекс FAISS
import faiss
index = faiss.IndexFlatIP(chunk_vecs.shape[1])
index.add(chunk_vecs)
print("Индекс FAISS построен")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 5556.59it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Бэкенд эмбеддингов: STBackend
Размерность эмбеддингов: (18, 384)
Индекс FAISS построен


In [6]:
def search(query: str, top_k: int = 5) -> pd.DataFrame:
    qvec = embedder.encode([query]).astype("float32")
    scores, idx = index.search(qvec, top_k)
    res = chunks_df.iloc[idx[0]].copy().reset_index(drop=True)
    res["rank"] = range(1, len(res)+1)
    res["score"] = scores[0]
    return res[["rank","score","doc_id","title","chunk_id","chunk_text"]]

# Тестовые запросы
for q in ["Что такое модель OSI?", "Как работает маршрутизация?", "Как обнаружить потерю пакетов?"]:
    display(Markdown(f"**Запрос:** {q}"))
    display(search(q, top_k=3))

**Запрос:** Что такое модель OSI?

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.684125,net_01,Модель OSI,net_01_c0,Модель OSI состоит из семи уровней: физический...
1,2,0.445214,net_08,Протоколы прикладного уровня,net_08_c0,"HTTP, DNS, SMTP и FTP работают на прикладном у..."
2,3,0.288410,net_04,Маршрутизация в сетях,net_04_c0,Маршрутизаторы анализируют заголовки пакетов и...


**Запрос:** Как работает маршрутизация?

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.627942,net_02,Стек протоколов TCP/IP,net_02_c1,обеспечивают надёжную доставку пакетов и маршр...
1,2,0.617598,net_04,Маршрутизация в сетях,net_04_c0,Маршрутизаторы анализируют заголовки пакетов и...
2,3,0.559976,net_09,Диагностика сетей,net_09_c0,"Утилиты ping, traceroute и tcpdump помогают вы..."


**Запрос:** Как обнаружить потерю пакетов?

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.516186,net_03,Ошибки передачи данных,net_03_c0,"При передаче данных возможны потери пакетов, и..."
1,2,0.457233,net_09,Диагностика сетей,net_09_c0,"Утилиты ping, traceroute и tcpdump помогают вы..."
2,3,0.253489,net_08,Протоколы прикладного уровня,net_08_c1,"и ответов для веб-серфинга, почты и передачи ф..."


In [7]:
# Контрольные запросы (10 шт.) с ожидаемыми релевантными doc_id
benchmark = [
    {"qid":"q1","query":"Сколько уровней в модели OSI?","expected":["net_01"]},
    {"qid":"q2","query":"Какие протоколы входят в TCP/IP?","expected":["net_02"]},
    {"qid":"q3","query":"Как обнаруживаются ошибки передачи?","expected":["net_03"]},
    {"qid":"q4","query":"Что делают маршрутизаторы?","expected":["net_04"]},
    {"qid":"q5","query":"Как приоритезировать голосовой трафик?","expected":["net_05"]},
    {"qid":"q6","query":"Как защитить сеть от перехвата?","expected":["net_06"]},
    {"qid":"q7","query":"Какие проблемы у беспроводных сетей?","expected":["net_07"]},
    {"qid":"q8","query":"Какие протоколы прикладного уровня вы знаете?","expected":["net_08"]},
    {"qid":"q9","query":"Какими утилитами диагностировать сеть?","expected":["net_09"]},
    {"qid":"q10","query":"Как масштабировать сеть при росте нагрузки?","expected":["net_10"]},
]

def eval_query(q: str, expected: List[str], top_k: int = 5) -> Dict:
    res = search(q, top_k)
    retrieved = res["doc_id"].tolist()
    hit = int(any(e in retrieved for e in expected))
    recall = sum(e in retrieved for e in expected) / len(expected)
    first_rank = next((i+1 for i,d in enumerate(retrieved) if d in expected), None)
    mrr = 1/first_rank if first_rank else 0
    return {"retrieved": retrieved, "hit": hit, "recall": recall, "mrr": mrr, "first_rank": first_rank}

# Оценка
eval_rows = []
for b in benchmark:
    m = eval_query(b["query"], b["expected"], top_k=5)
    eval_rows.append({
        "query": b["query"],
        "expected_source": ",".join(b["expected"]),
        "retrieved_sources": ",".join(m["retrieved"]),
        "hit_at_5": m["hit"],
        "recall_at_5": m["recall"],
        "mrr_at_5": m["mrr"],
        "rank_of_first_relevant": m["first_rank"]
    })
eval_df = pd.DataFrame(eval_rows)

# Сохраняем артефакт
eval_df.to_csv("artifacts/retrieval_eval.csv", index=False, encoding="utf-8-sig")
print(" Сохранено: artifacts/retrieval_eval.csv")
display(eval_df)

# Сводные метрики
print(f"\nСредние метрики (k=5):")
print(f"  hit@5: {eval_df['hit_at_5'].mean():.3f}")
print(f"  recall@5: {eval_df['recall_at_5'].mean():.3f}")
print(f"  MRR@5: {eval_df['mrr_at_5'].mean():.3f}")

 Сохранено: artifacts/retrieval_eval.csv


,query,expected_source,retrieved_sources,hit_at_5,recall_at_5,mrr_at_5,rank_of_first_relevant
0,Сколько уровней в модели OSI?,net_01,"net_01,net_08,net_01,net_10,net_07",1,1.0,1.0,1
1,Какие протоколы входят в TCP/IP?,net_02,"net_02,net_09,net_02,net_08,net_06",1,1.0,1.0,1
2,Как обнаруживаются ошибки передачи?,net_03,"net_03,net_09,net_07,net_09,net_07",1,1.0,1.0,1
3,Что делают маршрутизаторы?,net_04,"net_04,net_02,net_09,net_07,net_02",1,1.0,1.0,1
4,Как приоритезировать голосовой трафик?,net_05,"net_05,net_07,net_07,net_10,net_09",1,1.0,1.0,1
5,Как защитить сеть от перехвата?,net_06,"net_06,net_09,net_07,net_02,net_02",1,1.0,1.0,1
6,Какие проблемы у беспроводных сетей?,net_07,"net_07,net_09,net_02,net_10,net_02",1,1.0,1.0,1
7,Какие протоколы прикладного уровня вы знаете?,net_08,"net_07,net_04,net_02,net_02,net_08",1,1.0,0.2,5
8,Какими утилитами диагностировать сеть?,net_09,"net_09,net_04,net_08,net_01,net_02",1,1.0,1.0,1
9,Как масштабировать сеть при росте нагрузки?,net_10,"net_10,net_02,net_02,net_01,net_04",1,1.0,1.0,1



Средние метрики (k=5):
  hit@5: 1.000
  recall@5: 1.000
  MRR@5: 0.920


In [8]:
# Сравниваем два варианта chunk_size: 15 и 25 (overlap = 5)
def build_and_eval(chunk_size: int, overlap: int = 5) -> float:
    ch_df = build_chunks_df(documents, chunk_size, overlap)
    vecs = embedder.fit(ch_df["chunk_text"].tolist())
    idx = faiss.IndexFlatIP(vecs.shape[1]); idx.add(vecs)
    hits = 0
    for b in benchmark:
        qvec = embedder.encode([b["query"]]).astype("float32")
        _, ids = idx.search(qvec, 5)
        retrieved = [ch_df.iloc[i]["doc_id"] for i in ids[0]]
        if any(e in retrieved for e in b["expected"]): hits += 1
    return hits / len(benchmark)

exp_results = []
for cs in [15, 25]:
    h = build_and_eval(cs)
    exp_results.append({"chunk_size": cs, "hit@5": h})
    print(f"chunk_size={cs} → hit@5 = {h:.3f}")

exp_df = pd.DataFrame(exp_results)
print("\nСравнение конфигураций чанкинга:")
display(exp_df)

chunk_size=15 → hit@5 = 1.000
chunk_size=25 → hit@5 = 1.000

Сравнение конфигураций чанкинга:


,chunk_size,hit@5
0,15,1.0
1,25,1.0


In [9]:
# Добавляем 3 новых документа
new_docs = [
    {"doc_id": "net_11", "title": "SDN и виртуализация", "text": "Программно-определяемые сети (SDN) отделяют плоскость управления от плоскости данных. Это позволяет централизованно управлять сетью и быстро адаптировать конфигурации под меняющиеся требования."},
    {"doc_id": "net_12", "title": "IoT в телекоммуникациях", "text": "Устройства Интернета вещей генерируют огромный объём данных. Для их передачи используются специализированные протоколы (MQTT, CoAP) и энергоэффективные технологии (LoRaWAN, NB-IoT)."},
    {"doc_id": "net_13", "title": "5G и низкие задержки", "text": "Пятое поколение сотовой связи обеспечивает задержки менее 1 мс. Это критично для автономного транспорта, удалённой хирургии и промышленной автоматизации, где важна мгновенная реакция."},
]
updated_docs = documents + new_docs

# Функция для оценки retrieval на расширенном бенчмарке
def evaluate_retrieval(docs: List[Dict], queries: List[Dict], top_k: int = 5) -> pd.DataFrame:
    ch_df = build_chunks_df(docs, 20, 5)
    vecs = embedder.fit(ch_df["chunk_text"].tolist())
    idx = faiss.IndexFlatIP(vecs.shape[1]); idx.add(vecs)
    rows = []
    for q in queries:
        qvec = embedder.encode([q["query"]]).astype("float32")
        _, ids = idx.search(qvec, top_k)
        retrieved = [ch_df.iloc[i]["doc_id"] for i in ids[0]]
        rows.append({"query": q["query"], "retrieved": ",".join(retrieved)})
    return pd.DataFrame(rows)

# Расширяем бенчмарк запросами к новым документам
extended_bench = benchmark + [
    {"qid":"q11","query":"Что такое SDN и зачем она нужна?","expected":["net_11"]},
    {"qid":"q12","query":"Какие протоколы используют IoT-устройства?","expected":["net_12"]},
    {"qid":"q13","query":"Какие преимущества у 5G для промышленности?","expected":["net_13"]},
]

# Оценка ДО обновления (только оригинальные 10 документов)
before_df = evaluate_retrieval(documents, extended_bench)
# Оценка ПОСЛЕ обновления (13 документов)
after_df = evaluate_retrieval(updated_docs, extended_bench)

# Сравнение
comp_df = before_df.merge(after_df, on="query", suffixes=("_before","_after"))
comp_df["changed"] = comp_df["retrieved_after"] != comp_df["retrieved_before"]

# Сохраняем артефакт
comp_df.to_csv("artifacts/retrieval_before_after_update.csv", index=False, encoding="utf-8-sig")
print(" Сохранено: artifacts/retrieval_before_after_update.csv")
display(comp_df[comp_df["changed"]].head(5) if comp_df["changed"].any() else comp_df.head(5))

 Сохранено: artifacts/retrieval_before_after_update.csv


,query,retrieved_before,retrieved_after,changed
1,Какие протоколы входят в TCP/IP?,"net_02,net_09,net_02,net_08,net_06","net_02,net_09,net_12,net_02,net_08",True
4,Как приоритезировать голосовой трафик?,"net_05,net_07,net_07,net_10,net_09","net_05,net_07,net_13,net_07,net_10",True
6,Какие проблемы у беспроводных сетей?,"net_07,net_09,net_02,net_10,net_02","net_07,net_09,net_02,net_12,net_10",True
8,Какими утилитами диагностировать сеть?,"net_09,net_04,net_08,net_01,net_02","net_09,net_04,net_08,net_12,net_01",True
9,Как масштабировать сеть при росте нагрузки?,"net_10,net_02,net_02,net_01,net_04","net_10,net_11,net_12,net_02,net_02",True


In [10]:
def build_context(query: str, top_k: int = 3) -> Tuple[str, pd.DataFrame]:
    res = search(query, top_k)
    ctx = "\n\n".join([f"[{r['doc_id']}|{r['title']}] {r['chunk_text']}" for _,r in res.iterrows()])
    return ctx, res

def generate_answer(query: str, context: str) -> str:
    # Простой extractive-генератор: выбираем предложение, наиболее близкое к запросу
    sentences = [s.strip() for s in re.split(r'[.!?]', context) if len(s.strip())>10]
    if not sentences: return "Недостаточно контекста для ответа."
    vec = TfidfVectorizer(ngram_range=(1,2)).fit_transform([query]+sentences).toarray()
    scores = (vec[1:] @ vec[0]) / (np.linalg.norm(vec[1:],axis=1)*np.linalg.norm(vec[0])+1e-12)
    best = sentences[np.argmax(scores)]
    return best if len(best.split())>=4 else sentences[0]

def mini_rag(query: str, top_k: int = 3) -> Dict:
    ctx, src = build_context(query, top_k)
    ans = generate_answer(query, ctx)
    return {
        "question": query, 
        "answer": ans, 
        "retrieved_sources": src[["doc_id","title","chunk_text"]].to_dict("records"), 
        "context": ctx
    }

# Примеры работы mini-RAG
rag_examples = []
for q in ["Как работает модель OSI?", "Как защитить сеть?", "Что даёт 5G?"]:
    r = mini_rag(q)
    rag_examples.append({
        "question": q, 
        "answer": r["answer"], 
        "retrieved_sources": "; ".join([f"{s['doc_id']}:{s['title']}" for s in r["retrieved_sources"]])
    })
    display(Markdown(f"**Вопрос:** {q}\n\n**Ответ:** {r['answer']}\n\n**Источники:**"))
    display(pd.DataFrame(r["retrieved_sources"]))

# Сохраняем артефакт
pd.DataFrame(rag_examples).to_csv("artifacts/rag_examples.csv", index=False, encoding="utf-8-sig")
print(" Сохранено: artifacts/rag_examples.csv")

**Вопрос:** Как работает модель OSI?

**Ответ:** [net_01|Модель OSI] Модель OSI состоит из семи уровней: физический, канальный, сетевой, транспортный, сеансовый, представления и прикладной

**Источники:**

,doc_id,title,chunk_text
0,net_01,Модель OSI,Модель OSI состоит из семи уровней: физический...
1,net_08,Протоколы прикладного уровня,"HTTP, DNS, SMTP и FTP работают на прикладном у..."
2,net_07,Беспроводные технологии,применяются методы модуляции и множественного ...


**Вопрос:** Как защитить сеть?

**Ответ:** [net_06|Безопасность сетей] Межсетевые экраны, VPN и шифрование TLS защищают данные от перехвата и несанкционированного доступа

**Источники:**

,doc_id,title,chunk_text
0,net_06,Безопасность сетей,"Межсетевые экраны, VPN и шифрование TLS защища..."
1,net_07,Беспроводные технологии,"Wi-Fi, Bluetooth и сотовые сети используют рад..."
2,net_02,Стек протоколов TCP/IP,обеспечивают надёжную доставку пакетов и маршр...


**Вопрос:** Что даёт 5G?

**Ответ:** [net_07|Беспроводные технологии] применяются методы модуляции и множественного доступа

**Источники:**

,doc_id,title,chunk_text
0,net_07,Беспроводные технологии,применяются методы модуляции и множественного ...
1,net_07,Беспроводные технологии,"Wi-Fi, Bluetooth и сотовые сети используют рад..."
2,net_02,Стек протоколов TCP/IP,"TCP/IP включает четыре уровня: прикладной, тра..."


 Сохранено: artifacts/rag_examples.csv


In [11]:
display(Markdown("## Анализ неудачных случаев mini-RAG"))

weak_cases = []
for b in benchmark[:5]:
    r = mini_rag(b["query"])
    ans = r["answer"].lower()
    expected_kw = {
        "net_01": ["os", "уровен", "модель"],
        "net_02": ["tcp", "ip", "протокол"],
        "net_03": ["ошибк", "crc", "повтор"],
        "net_04": ["маршрут", "таблиц", "ospf"],
        "net_05": ["приоритет", "трафик", "qos"],
    }
    kw_list = []
    for exp in b["expected"]:
        kw_list.extend(expected_kw.get(exp, []))
    if kw_list and not any(kw in ans for kw in kw_list):
        weak_cases.append({"query": b["query"], "answer": r["answer"], "expected_kw": kw_list})

if weak_cases:
    for wc in weak_cases:
        display(Markdown(f"**Запрос:** {wc['query']}\n\n**Ответ:** {wc['answer']}\n\n**Ожидаемые ключевые слова:** {', '.join(wc['expected_kw'])}\n\n**Возможная причина:** генератор выбрал предложение, лексически близкое к запросу, но не содержащее фактического ответа."))
else:
    print(" На учебном наборе все ответы содержат ожидаемые ключевые слова.")

## Анализ неудачных случаев mini-RAG

 На учебном наборе все ответы содержат ожидаемые ключевые слова.


In [12]:
display(Markdown("""## Итоги выполнения HW14

 Загружена база знаний по телекоммуникациям (10 документов)  
 Реализован чанкинг (chunk_size=20, overlap=5)  
 Построены эмбеддинги и индекс FAISS  
 Выполнен retrieval по 10 контрольным запросам  
 Рассчитаны метрики: hit@5, recall@5, MRR@5  
 Проведён эксперимент с chunk_size  
 Добавлено 3 новых документа, выполнена переиндексация и сравнение  
 Собран mini-RAG с возвратом источников  
 Проанализированы слабые случаи  
 Все артефакты сохранены в `artifacts/`"""))

## Итоги выполнения HW14

 Загружена база знаний по телекоммуникациям (10 документов)  
 Реализован чанкинг (chunk_size=20, overlap=5)  
 Построены эмбеддинги и индекс FAISS  
 Выполнен retrieval по 10 контрольным запросам  
 Рассчитаны метрики: hit@5, recall@5, MRR@5  
 Проведён эксперимент с chunk_size  
 Добавлено 3 новых документа, выполнена переиндексация и сравнение  
 Собран mini-RAG с возвратом источников  
 Проанализированы слабые случаи  
 Все артефакты сохранены в `artifacts/`